# 07 — Factibilidad de historia extendida y aprendizaje cross-asset

> Pregunta: ¿podemos construir un panel desde 2010 con suficiente diversidad económica y calidad para estudiar modelos temporales compartidos? Esta notebook es un diagnóstico local: no descarga datos, no altera Gold y no entrena modelos.

El inventario distingue evidencia disponible en el repositorio de cobertura que todavía debe adquirirse y verificarse.

## <b>Block 01 — Inventario Gold actual</b>

> Las siete series del experimento cross-asset son SPY, QQQ, IWM, IWD, IWF, XLE y XLF. Se revisan directamente desde Gold con las mismas columnas que alimentan las notebooks 01–06.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'Momentum_systematic_strategies').exists():
    PROJECT = next(parent for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (parent / 'Momentum_systematic_strategies').exists())
GOLD = PROJECT / 'Momentum_systematic_strategies' / 'data' / 'gold' / 'daily_series' / 'etf'
UNIVERSE = ['SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF']
EXPECTED = {'symbol_requested', 'session_date', 'open', 'high', 'low', 'close', 'adjusted_close', 'volume',
            'dividend_amount', 'split_coefficient', 'provider_timezone', 'source',
            'provider_last_refreshed', 'ingested_at_utc'}

def audit_gold(symbol):
    frame = pl.read_parquet(GOLD / f'{symbol}.parquet').sort('session_date')
    data = frame.to_dict(as_series=False)
    n = len(data['session_date'])
    dates = pd.to_datetime(data['session_date'])
    bad_price = sum(any(data[column][i] is None or data[column][i] <= 0
                        for column in ['open', 'high', 'low', 'close', 'adjusted_close']) for i in range(n))
    bad_ohlc = sum(data['high'][i] < max(data['open'][i], data['close'][i], data['low'][i])
                   or data['low'][i] > min(data['open'][i], data['close'][i], data['high'][i])
                   for i in range(n))
    return {
        'symbol': symbol, 'start_date': dates.min().date(), 'end_date': dates.max().date(), 'n_sessions': n,
        'adjusted_close_nulls': int(sum(value is None for value in data['adjusted_close'])),
        'duplicate_symbol_date': int(frame.select(pl.struct(['symbol_requested', 'session_date']).is_duplicated().sum()).item()),
        'nonpositive_ohlc_or_adj': bad_price, 'ohlc_violations': bad_ohlc,
        'dividend_dates': int(sum(value is not None and value > 0 for value in data['dividend_amount'])),
        'split_dates': int(sum(value is not None and value != 1 for value in data['split_coefficient'])),
        'source': ', '.join(sorted(set(data['source']))),
        'schema_complete': EXPECTED.issubset(frame.columns),
        'dates': set(data['session_date']),
    }

audits = [audit_gold(symbol) for symbol in UNIVERSE]
audit_table = pd.DataFrame(audits).drop(columns='dates')
display(audit_table)

common_dates = set.intersection(*(row['dates'] for row in audits))
print(f'Calendario común de las siete series: {min(common_dates)} to {max(common_dates)} | {len(common_dates):,} sesiones')
print('Sesiones con historia local anterior a 2010:', sum(row['start_date'].year < 2010 for row in audits))

La disponibilidad local demuestra una muestra común desde 2018, no desde 2010. `adjusted_close` es completo en las siete series; los dividendos y splits se conservan como campos de procedencia. Gold contiene también `GLD.parquet`, que se revisa como posible proxy de oro, pero no forma parte del universo de las notebooks 01–06.

## <b>Block 02 — Calidad, calendario y procedencia</b>

> El pipeline existente promueve Silver a Gold después de controles de esquema, duplicados, orden temporal, OHLC, precios positivos y eventos corporativos. Esta revisión verifica sus invariantes en los archivos disponibles; no sustituye una auditoría por instrumento de una ampliación futura.

In [ ]:
quality_controls = pd.DataFrame([
    ('Esquema y procedencia', '14 campos esperados; símbolo, fuente y marcas de refresco conservados'),
    ('Fechas', 'sin duplicados symbol/date, orden ascendente y sin fechas futuras'),
    ('OHLC y volumen', 'precios estrictamente positivos; high/low coherentes; volumen no negativo'),
    ('Corporate actions', 'dividend_amount no negativo y split_coefficient positivo; eventos visibles para revisión'),
    ('Sesiones', 'calendario consenso como alerta; para una extensión se requiere contraste con calendario de mercado'),
    ('Retornos', 'retornos de investigación sobre adjusted_close; no imputar sesiones ni precios faltantes'),
], columns=['control', 'regla existente o requerida'])
display(quality_controls)

gap_rows = []
for row in audits:
    dates = sorted(pd.to_datetime(list(row['dates'])))
    gaps_over_four_days = int((pd.Series(dates).diff().dt.days > 4).sum())
    gap_rows.append((row['symbol'], gaps_over_four_days))
display(pd.DataFrame(gap_rows, columns=['symbol', 'calendar_gaps_over_4_days']))
print('Nota: un hueco de más de cuatro días es sólo una alerta descriptiva; la validación final debe usar un calendario oficial del mercado aplicable.')

## <b>Block 03 — Cobertura 2010 y diversidad económica</b>

> No se consulta ningún proveedor en esta notebook. El catálogo local confirma mapeos ETF elegibles, mientras que la presencia física de un Parquet Gold confirma sólo lo que está disponible hoy. Por tanto, una entrada sin Gold no es una afirmación de ausencia en el proveedor: requiere adquisición y verificación autorizadas.

In [ ]:
catalog = pd.read_csv(PROJECT / 'Momentum_systematic_strategies' / 'data_ingest' / 'daily_universe.csv')
candidates = pd.DataFrame([
    ('SPY', 'mercado amplio EE. UU.', 'universo actual'), ('QQQ', 'growth / tecnología', 'universo actual'),
    ('IWM', 'small caps', 'universo actual'), ('IWD', 'value', 'universo actual'),
    ('IWF', 'growth', 'universo actual'), ('XLE', 'energía', 'universo actual'),
    ('XLF', 'financieras', 'universo actual'), ('GLD', 'oro', 'Gold local adicional'),
    ('TLT', 'bonos soberanos EE. UU. larga duración', 'candidato'), ('SHY', 'bonos soberanos EE. UU. corta duración', 'candidato'),
    ('BIL', 'letras del Tesoro / efectivo', 'candidato'), ('IAU', 'oro alternativo', 'candidato'),
    ('USO', 'petróleo como proxy de materias primas', 'candidato'),
], columns=['symbol', 'proxy_económico', 'rol'])
catalog_status = (catalog.drop_duplicates('ticker_alpha_vantage').set_index('ticker_alpha_vantage')
                  [['identity_status', 'download_decision']].to_dict('index'))
candidates['parquet_gold_local'] = candidates['symbol'].map(lambda symbol: (GOLD / f'{symbol}.parquet').exists())
candidates['catálogo_local'] = candidates['symbol'].map(lambda symbol: catalog_status.get(symbol, {}).get('identity_status', 'no listado'))
candidates['estado_2010'] = np.where(
    candidates['parquet_gold_local'],
    candidates['symbol'].map({row['symbol']: f"Gold local desde {row['start_date']}" for row in audits + [audit_gold('GLD')]}),
    'requiere adquisición y validación; cobertura no comprobada aquí'
)
display(candidates)

print('Conclusión de disponibilidad: no hay evidencia local de un panel 2010 para las siete series actuales ni para GLD.')

## <b>Block 04 — Protocolo de ampliación reproducible</b>

> La ampliación debe ser una nueva versión de datos, no una reescritura silenciosa de Gold. La historia actual queda inmóvil y cada candidato se descarga a Silver, se audita y sólo se promueve tras revisión explícita.

In [ ]:
protocol = pd.DataFrame([
    (1, 'Congelar inventario', 'registrar hash, cobertura y schema de cada Gold actual; no truncar ni reemplazar archivos existentes'),
    (2, 'Manifest de candidatos', 'definir símbolo, clase ETF, proxy económico, fuente esperada y cobertura mínima 2010 antes de descargar'),
    (3, 'Ingesta aislada', 'con autorización, usar el pipeline diario hacia Silver; no analizar raw ni Silver como fuente de investigación'),
    (4, 'Quality review', 'aplicar reglas existentes y contraste de sesiones, adjusted_close y eventos corporativos contra la procedencia del proveedor'),
    (5, 'Promoción versionada', 'promover atómicamente a Gold sólo las series aprobadas y conservar un manifiesto con hashes y fechas'),
    (6, 'Panel de investigación', 'usar la intersección efectiva; excluir instrumentos con cobertura o eventos no verificables, sin imputación'),
], columns=['paso', 'etapa', 'resultado reproducible'])
display(protocol)

print('Comando previsto, no ejecutado:')
print('python Momentum_systematic_strategies/data_ingest/run_daily_pipeline.py --symbol <SYMBOL> --asset-class etf --expected-session-date <LAST_COMPLETED_US_SESSION>')

### <b>Diseño temporal que habilitaría una cobertura común 2010+</b>

Con una intersección aprobada desde 2010, se reservaría primero el tramo necesario para indicadores y secuencias causales: 126 sesiones para momentum, volatilidad EWMA y 63 sesiones de contexto. Un diseño inicial, todavía no ejecutado, podría entrenar de la primera decisión válida hasta 2018, usar 2019–2020 para validación de diseño y hacer walk-forward anual desde 2021. Cada corte conservaría la regla de que la posición de $t$ sólo recibe el retorno de $t+1$.

La ampliación añadiría duración, efectivo y oro —y eventualmente materias primas— frente al panel actual de renta variable estadounidense. Los periodos concretos sólo se congelarán después de conocer la intersección Gold realmente aprobada; no se debe forzar una ventana si una serie no la cubre.

## <b>Interpretación y recomendación</b>

- La infraestructura local ya es suficiente para una ampliación controlada: el esquema Gold, el pipeline diario y los controles de promoción existen.
- El panel compartido actual es homogéneo en renta variable estadounidense y comienza en 2018; no sustenta por sí solo una afirmación de historia desde 2010.
- GLD está disponible localmente como proxy de oro, pero su archivo actual también comienza en 2018. Bonos y materias primas propuestos están confirmados en el catálogo, no en Gold.
- Si se verifica un panel 2010+ con renta variable, duración corta/larga, oro y materias primas, la nueva hipótesis será que una corrección compartida aprende patrones temporales que se transfieren entre fuentes de riesgo, no sólo entre ETFs correlacionados de acciones.
- Los controles mínimos son split temporal por activos y fechas, normalización ajustada sólo en entrenamiento, evaluación leave-one-asset-out, comparación contra TSMOM y reporte separado por clase de activo.

La recomendación es **GO para planificar y auditar una ampliación versionada**, pero **NO-GO para entrenar modelos 2010+** hasta adquirir con autorización, validar y promover una cobertura común suficiente.